In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
import seaborn as sns

In [2]:
df = pd.read_csv("data//fraud_oracle_with_text.csv")

In [3]:
df.sample()

,Month,WeekOfMonth,DayOfWeek,Make,AccidentArea,DayOfWeekClaimed,MonthClaimed,WeekOfMonthClaimed,Sex,MaritalStatus,...,AgeOfPolicyHolder,PoliceReportFiled,WitnessPresent,AgentType,NumberOfSuppliments,AddressChange_Claim,NumberOfCars,Year,BasePolicy,ClaimNarrative
14801,Sep,5,Thursday,Pontiac,Urban,Monday,Oct,1,Male,Single,...,31 to 35,Yes,No,Internal,none,no change,1 vehicle,1996,Liability,NaN


In [4]:
df_text = df[df['ClaimNarrative'].notna()]

In [6]:
df_text.sample(5)

,Month,WeekOfMonth,DayOfWeek,Make,AccidentArea,DayOfWeekClaimed,MonthClaimed,WeekOfMonthClaimed,Sex,MaritalStatus,...,AgeOfPolicyHolder,PoliceReportFiled,WitnessPresent,AgentType,NumberOfSuppliments,AddressChange_Claim,NumberOfCars,Year,BasePolicy,ClaimNarrative
383,Oct,5,Friday,VW,Urban,Friday,Oct,5,Male,Married,...,over 65,No,No,External,none,no change,1 vehicle,1994,All Perils,"On a Friday in the fifth week of October, a th..."
433,Jun,3,Sunday,Toyota,Rural,Wednesday,Jul,2,Male,Married,...,41 to 50,No,No,External,none,no change,1 vehicle,1994,Collision,"On a Sunday in the third week of June, the pol..."
385,Jan,1,Tuesday,Honda,Urban,Wednesday,Jan,4,Male,Single,...,31 to 35,No,No,External,none,no change,1 vehicle,1994,All Perils,On a Tuesday in January during the first week ...
388,Jan,1,Tuesday,Honda,Urban,Tuesday,Jan,1,Male,Single,...,16 to 17,No,No,External,none,no change,1 vehicle,1994,Liability,On a Tuesday in January during the first week ...
289,Jan,1,Thursday,Chevrolet,Urban,Friday,Jan,2,Male,Married,...,36 to 40,No,No,External,more than 5,no change,1 vehicle,1994,Liability,On a Thursday in January during the first week...


In [9]:
X= df_text['ClaimNarrative']
y=df_text["FraudFound_P"]

In [10]:
X

0      On a Wednesday in December of week five, the p...
1      On a Wednesday in the third week of January, t...
2      The policy holder was involved in a collision ...
3      On a Saturday in June during the second week o...
4      On a Monday in late January, an urban collisio...
                             ...                        
488    On a Friday in January during the second week ...
489    On a Monday in October during the second week ...
490    On a Thursday in June during the third week of...
491    On a Wednesday in the third week of June, a co...
492    On a Monday in the second week of May, the pol...
Name: ClaimNarrative, Length: 491, dtype: object

In [12]:
y.value_counts()

FraudFound_P
0    461
1     30
Name: count, dtype: int64

In [13]:
from sklearn.model_selection import train_test_split

X_train_text, X_test_text, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [14]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2)
)

X_train_tfidf = tfidf.fit_transform(X_train_text)
X_test_tfidf = tfidf.transform(X_test_text)


In [15]:
X_train_tfidf.shape

(392, 2035)

In [17]:
X_test_tfidf.shape

(99, 2035)

In [18]:
from sklearn.linear_model import LogisticRegression

text_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

text_model.fit(
    X_train_tfidf,
    y_train
)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,42
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [19]:
y_pred = text_model.predict(X_test_tfidf)

In [20]:
from sklearn.metrics import *

In [21]:
accuracy_score(y_test, y_pred)

0.9090909090909091

In [22]:
precision_score(y_test, y_pred)

0.2

In [23]:
recall_score(y_test, y_pred)

0.16666666666666666

In [24]:
f1_score(y_test, y_pred)

0.18181818181818182

In [34]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import cross_val_score,RandomizedSearchCV,cross_validate

In [35]:
models = {
    'Decision Tree': DecisionTreeClassifier(random_state = 42),
    'Random forest': RandomForestClassifier(random_state = 42),
    'XGBoost': XGBClassifier(random_state = 42)
}

In [40]:
cv_results = {}
scoring_metrics = ['accuracy', 'precision_weighted', 'recall_weighted', 'f1_weighted']
for model_name , model in models.items():
    print(f'{model_name} is ruuning ')
    scores = cross_validate(
        model, 
        X_train_tfidf, 
        y_train, 
        cv=5, 
        scoring=scoring_metrics,
        return_train_score=False
    )
    
    cv_results[model_name] = scores
    
    # Print the mean scores
    print(f"{model_name} Results:")
    print(f"  Accuracy:  {np.mean(scores['test_accuracy']):.4f}")
    print(f"  Precision: {np.mean(scores['test_precision_weighted']):.4f}")
    print(f"  Recall:    {np.mean(scores['test_recall_weighted']):.4f}")
    print(f"  F1-Score:  {np.mean(scores['test_f1_weighted']):.4f}")
    print('-' * 50)

Decision Tree is ruuning 
Decision Tree Results:
  Accuracy:  0.9107
  Precision: 0.9046
  Recall:    0.9107
  F1-Score:  0.9074
--------------------------------------------------
Random forest is ruuning 


C:\Users\ASHISH\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\ASHISH\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\ASHISH\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(

Random forest Results:
  Accuracy:  0.9388
  Precision: 0.8813
  Recall:    0.9388
  F1-Score:  0.9092
--------------------------------------------------
XGBoost is ruuning 


C:\Users\ASHISH\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


XGBoost Results:
  Accuracy:  0.9286
  Precision: 0.8808
  Recall:    0.9286
  F1-Score:  0.9040
--------------------------------------------------


C:\Users\ASHISH\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
